# Stage 2: Train, Ablate & Evaluate Aerial Vehicle Detector (`02_train_and_evaluate_detector.ipynb`)

### End-to-End Deep Learning Pipeline for thefourthlaw.ai Test Task

This notebook trains and evaluates an ultra-lightweight neural detector under strict industrial and real-world constraints:
- **Rule 1 (Data Split Isolation)**: Video-level isolation. `eval` (`32179597.mp4`) and `eval_B` (`13763017.mp4`) are strictly held-out in `test` split. Zero test frames are ever seen during training or threshold calibration.
- **Rule 4 (Optical Distance Bands)**: Evaluated across `0–200 m` (near) and `200–400 m` (far) using pinhole camera optics and vehicle physical dimensions.
- **Rule 5 (Edge Model Constraints)**: Custom **YOLO26-Pico-P2** architecture (< 500,000 parameters, P2/4–P3/8–P4/16 high-resolution pyramid, NMS-free dual assignment, DFL-free regression).
- **Rule 6 (Scientific Ablation Study)**: Systematic comparison between `exp1_mosaic` (standard square training with mosaic) vs `exp2_rect_nomosaic` (aspect-ratio preserving rectangular training without mosaic to prevent small-vehicle degradation at 200–400 m).
- **Champion Guard & Append-Only Model Registry**: Non-overwriting immutable model versions in `weights/registry/` protected by validation selection score.

In [ ]:
# === SECTION 1: ENVIRONMENT SETUP & DATASET SYNCHRONIZATION ===
import os
import sys
from pathlib import Path
import torch

GITHUB_REPO_URL = "https://github.com/ituvtu/aerial-vehicle-detector.git"
REPO_DIR = Path("/content/aerial-vehicle-detector")

if not REPO_DIR.exists():
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}

!pip install -q "ultralytics>=8.4.0" "git+https://github.com/ultralytics/CLIP.git" "pandas" "pyyaml" "matplotlib"

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi

import yaml
from src.manifest_manager import ensure_images_available, ensure_raw_eval_video_available, sync_manifest_splits

with open("configs/project_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ensure_images_available(
    manifest_path=cfg["paths"]["manifest_path"],
    images_dir=cfg["paths"]["images_dir"],
    bundle_zip_path=cfg["paths"]["images_bundle_zip"],
    bundle_url=cfg.get("remote_data", {}).get("images_bundle_url"),
)
split_counts = sync_manifest_splits()
print("Frames unpacked. Split distribution:", split_counts)

# Ensure raw 30 FPS evaluation video is ready for high-fidelity demo generation
raw_eval_video = ensure_raw_eval_video_available("32179597")
print(f"Raw evaluation video ready: {raw_eval_video} ({raw_eval_video.stat().st_size / 1e6:.1f} MB)")

# Audit curated labels & verify Rule 4 distance coverage
!python scripts/03_audit_labels.py

In [ ]:
# === SECTION 2: MODEL ARCHITECTURE AUDIT (<500k params, Rule 5) ===
from src.metrics import inspect_and_verify_architecture

arch_report = inspect_and_verify_architecture(
    yaml_path="configs/yolo26_pico_p2.yaml",
    pretrained_weights="yolo26n.pt",
    output_report_path="results/model_architecture_report.json",
    max_params=500_000,
)

print(f"\nArchitecture: {arch_report['model_name']}")
print(f"Unfused Parameters: {arch_report['unfused_params']:,} (<500k limit)")
print(f"Fused Parameters:   {arch_report['fused_params']:,} (<500k limit)")
print(f"Strides:            {arch_report['strides']}")
print("Parameter Breakdown:", arch_report['blocks'])
print("Stem Weight Transfer:", arch_report['pretrained_transfer'])

In [ ]:
# === SECTION 3: ABLATION TRAINING (exp1_mosaic vs exp2_rect_nomosaic) ===
# Runs both experiments on A100 GPU and calibrates optimal threshold tau* on val split (Rule 1)
!python scripts/04_train.py --mode ablation --epochs 150 --batch 8 --imgsz 1024

from IPython.display import Markdown, display, Image
from pathlib import Path

val_table_p = Path("results/val_ablation_comparison.md")
if val_table_p.exists():
    display(Markdown(val_table_p.read_text(encoding="utf-8")))

curves_p = Path("results/val_ablation_comparison.png")
if curves_p.exists():
    display(Image(filename=str(curves_p)))

In [ ]:
# === SECTION 4: DECOUPLED EVALUATION & FULL-FPS VIDEO GENERATION ===
# Fully decoupled: test any checkpoint (or active champion) instantly without retraining!
SELECTED_EXPERIMENT = "auto"  # "auto" (uses Champion Guard winner), or "exp2_rect_nomosaic", "exp1_mosaic"

eval_cmd = "python scripts/05_evaluate.py"
if SELECTED_EXPERIMENT != "auto":
    eval_cmd += f" --weights weights/{SELECTED_EXPERIMENT}_best.pt --calibration weights/calibration_val_{SELECTED_EXPERIMENT}.json"

!{eval_cmd}

eval_summary_p = Path("results/eval_summary_table.md")
if eval_summary_p.exists():
    display(Markdown(eval_summary_p.read_text(encoding="utf-8")))

demo_video_p = Path("results/eval_demo_32179597_full_fps.mp4")
if demo_video_p.exists():
    print(f"\nFull 30.0 FPS Evaluation Video generated: {demo_video_p} ({demo_video_p.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# === SECTION 5: PACKAGE & DOWNLOAD DELIVERABLES BUNDLE ===
import datetime
import zipfile
from pathlib import Path

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bundle_zip = Path(f"colab_results_bundle_{timestamp}.zip")

include_dirs = [Path("weights"), Path("results"), Path("assets"), Path("data/registry")]

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        if folder.exists():
            for item in folder.rglob("*"):
                if item.is_file() and not item.name.endswith(".mp4") and item.stat().st_size < 50_000_000:
                    zf.write(item, arcname=item.as_posix())

print(f"Packaged deliverables bundle: {bundle_zip} ({bundle_zip.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(bundle_zip))
    print("Download triggered in browser.")
except (ImportError, Exception) as exc:
    print(f"Running locally or download skipped: {exc}. Bundle saved at {bundle_zip}")